In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
cars = pd.read_csv("../data/processed/cars_cleaned.csv")
cars.head()

,seller,offerType,price,abtest,vehicleType,gearbox,powerPS,model,kilometer,fuelType,brand,notRepairedDamage,Age
0,private,offer,4450,test,limousine,manual,150,3er,150000,diesel,bmw,no,15.25
1,private,offer,13299,control,suv,manual,163,xc_reihe,150000,diesel,volvo,no,13.50
2,private,offer,3200,test,bus,manual,101,touran,150000,diesel,volkswagen,no,15.92
3,private,offer,4500,control,small car,manual,86,ibiza,60000,petrol,seat,no,13.00
4,private,offer,18750,test,suv,automatic,185,xc_reihe,150000,diesel,volvo,no,10.92


In [3]:
cars_encoded = pd.get_dummies(cars, drop_first=True)
cars_encoded.shape

(42772, 306)

In [4]:
X = cars_encoded.drop('price', axis=1)
y = np.log(cars_encoded['price'])  # log transform

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [6]:
baseline_pred = np.mean(y_train)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, np.repeat(baseline_pred, len(y_test)))
)

baseline_rmse

np.float64(1.173776667493959)

In [7]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

lr_rmse, lr_r2

(np.float64(0.6377691363051097), 0.704772996380413)

In [8]:
rf = RandomForestRegressor(
    n_estimators=80,
    max_depth=25,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

rf_rmse, rf_r2

(np.float64(0.4845101262381682), 0.8296136505808217)

In [9]:
results = pd.DataFrame({
    "Model": ["Baseline", "Linear Regression", "Random Forest"],
    "RMSE": [baseline_rmse, lr_rmse, rf_rmse],
    "R2": [None, lr_r2, rf_r2]
})

results

,Model,RMSE,R2
0,Baseline,1.173777,NaN
1,Linear Regression,0.637769,0.704773
2,Random Forest,0.484510,0.829614


In [10]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance.head(10)

Age                      0.572986
powerPS                  0.257357
kilometer                0.047338
notRepairedDamage_yes    0.030299
vehicleType_cabrio       0.010967
model_transporter        0.007773
brand_volkswagen         0.007288
fuelType_petrol          0.004886
fuelType_diesel          0.004322
abtest_test              0.004291
dtype: float64